**Kernel**

Select **Python (JupyterNotebook .venv)** (`jupyternotebook-venv`). If you see `No module named 'dotenv'`, the notebook is still attached to another Python (often Homebrew) that does not have `python-dotenv` installed — change the kernel, do not reinstall into that global Python unless you intentionally want it.

In [2]:
# Load env variables and create client
# Searches upward from cwd for .env — finds AI_Learning/.env at repo root.
from dotenv import load_dotenv, find_dotenv
from anthropic import Anthropic

load_dotenv(find_dotenv())

client = Anthropic()
model = "claude-haiku-4-5"

In [ ]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [ ]:
def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "python" or "json" or "regex",
        "solution_criteria": "Key criteria for evaluating the solution"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [81]:
dataset = generate_dataset()

with open('dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)

In [ ]:
def run_prompt(test_case):
    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Response only with Python, JSON, or a plain regex
* Do not add any comments or commentary or explanation
"""
    
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"])
    return output

In [85]:
def grade_by_model(test_case, output):
    evaluate = f"""
You are an expert code reviewer. Evaluate this AI-generated solution.
    
Task: {test_case["task"]}

Solution: {output}
    
Criteria to use for evaluation: {test_case["solution_criteria"]}

Provide your evaluation as a structured JSON object with:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement  
- "reasoning": A concise explanation of your assessment
- "score": A number between 1-10

In JSON string values, escape every backslash twice (e.g. write \\\\d+ not \\d+ for regex patterns).

Example output:
```json
{{
    "strengths": ["Correctly solves the task", "Uses efficient code"],
    "weaknesses": ["Could use more comments", "Could be more efficient"],
    "reasoning": "The solution correctly solves the task but could use more comments and be more efficient.",
    "score": 8
}}
"""

    

    messages = []
    add_user_message(messages, evaluate)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [68]:
import ast
import json
import re

def validate_json(output):
    try:
        json.loads(output.strip())
        return 10  # valid JSON
    except:
        return 0   # invalid

def validate_python(output):
    try:
        ast.parse(output.strip())
        return 10  # valid Python syntax
    except SyntaxError:
        return 0

def validate_regex(output):
    try:
        re.compile(output.strip())
        return 10  # valid regex
    except re.error:
        return 0

def grade_syntax(output, test_case):
    
    # Pick syntax validator based on expected format
    format = test_case["format"].lower()
    if format == "json":
        return validate_json(output)
    elif format == "python":
        return validate_python(output)
    elif format == "regex":
        return validate_regex(output)

In [ ]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    grade = grade_by_model(test_case, output)
    
    model_score = grade["score"]
    reasoning = grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)
    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "reasoning": reasoning,
        "score": score
    }

In [70]:
from statistics import mean


def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")
    return results

In [86]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average score: 3.6666666666666665


In [56]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 Region Extractor\n\nHere's a Python function that extracts the AWS region from an S3 bucket URI:\n\n```python\nimport re\n\ndef extract_region_from_s3_uri(s3_uri: str) -> str:\n    \"\"\"\n    Extracts the AWS region from an S3 bucket URI.\n    \n    Args:\n        s3_uri (str): S3 URI in format 's3://bucket-name/path' or 's3://bucket-name'\n        \n    Returns:\n        str: AWS region code (e.g., 'us-west-2')\n        \n    Raises:\n        ValueError: If no region is found in the bucket name\n    \"\"\"\n    # Extract bucket name from S3 URI\n    match = re.match(r's3://([^/]+)', s3_uri)\n    if not match:\n        raise ValueError(f\"Invalid S3 URI format: {s3_uri}\")\n    \n    bucket_name = match.group(1)\n    \n    # Look for AWS region pattern at the end of bucket name\n    # AWS regions follow pattern: region-code like us-west-2, eu-central-1, etc.\n    region_pattern = r'(us|eu|ap|ca|sa|me|af)-(north|south|east|west|central|-\\w+)*-\\d+'\n    r